Rizqy Jauhary Atsaany  
235150300111038  
TKOM - Embedded Artificial Intelligence - B  

### **PENTING: HARUS PAKAI PYTHON 3.10 UNTUK WORK**

## 1. Latih model dasar (baseline model)

In [10]:
import tensorflow as tf
from tf_keras.models import Sequential
from tf_keras.layers import Dense, Flatten
from tf_keras.datasets import mnist
from tf_keras.utils import to_categorical

# Load data
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
y_train, y_test = to_categorical(y_train), to_categorical(y_test)

# Baseline model
model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.fit(x_train, y_train, epochs=2, validation_split=0.1)
loss, accuracy = model.evaluate(x_test, y_test)
print(f"\nAkurasi pada data uji: {accuracy:.4f}")


Epoch 1/2
1688/1688 [==============================] - 6s 3ms/step - loss: 0.2767 - accuracy: 0.9197 - val_loss: 0.1202 - val_accuracy: 0.9647
Epoch 2/2
313/313 [==============================] - 0s 1ms/step - loss: 0.1042 - accuracy: 0.9684

Akurasi pada data uji: 0.9684


## 2. Pruning model menggunakan TensorFlow Model Optimization Toolkit

In [ ]:
import tensorflow_model_optimization as tfmot

# Define pruning schedule
pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0,
        final_sparsity=0.5,
        begin_step=0,
        end_step=len(x_train) // 128 * 2  # 2 epochs
    )D
}

# Apply pruning
pruned_model = tfmot.sparsity.keras.prune_low_magnitude(model, **pruning_params)

# Re-compile and train
pruned_model.compile(optimizer='adam',
                     loss='categorical_crossentropy',
                     metrics=['accuracy'])

pruned_model.fit(x_train, y_train,
                 epochs=2,
                 batch_size=128,
                 validation_split=0.1,
                 callbacks=[tfmot.sparsity.keras.UpdatePruningStep()])
loss, accuracy = pruned_model.evaluate(x_test, y_test)
print(f"\nAkurasi pada data uji: {accuracy:.4f}")

Epoch 1/2
422/422 [==============================] - 3s 5ms/step - loss: 0.0765 - accuracy: 0.9774 - val_loss: 0.0862 - val_accuracy: 0.9765
Epoch 2/2
313/313 [==============================] - 1s 2ms/step - loss: 0.0803 - accuracy: 0.9760

Akurasi pada data uji: 0.9760


## 3. Post-Training Quantization (PTQ)

In [12]:
converter = tf.lite.TFLiteConverter.from_keras_model(pruned_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Aktivasi quantisasi
tflite_quant_model = converter.convert()

# Simpan model hasil quantization
with open('model_pruned_quantized.tflite', 'wb') as f:
    f.write(tflite_quant_model)


INFO:tensorflow:Assets written to: C:\Users\ASUSTU~1\AppData\Local\Temp\tmpb73dz4fk\assets


INFO:tensorflow:Assets written to: C:\Users\ASUSTU~1\AppData\Local\Temp\tmpb73dz4fk\assets


## 4. Cek Ukuran Model Sebelum dan Sesudah

In [13]:
import os

# Save original model
model.save('original_model.h5')

# Ukuran file
print("Original Model Size (MB):", os.path.getsize("original_model.h5") / 1e6)
print("Pruned + Quantized Model Size (MB):", os.path.getsize("model_pruned_quantized.tflite") / 1e6)


Original Model Size (MB): 1.247056
Pruned + Quantized Model Size (MB): 0.818308


d:\Rizqy\Kuliah\Sem 6\Embedded Artificial Intelligence\Repo\Embedded_AI\.venv\lib\site-packages\tf_keras\src\engine\training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
